# Chapter 8: Wrapping Up

## Chapter Summary
The final chapter connects modeling to problem formulation and deployment. A successful system needs a measurable goal, representative data, reproducible preprocessing, monitoring, tests, human oversight, and a plan for change after deployment.

## Theoretical Deep-Dive

An offline score is only a proxy for real-world value. Define the prediction target, decision threshold, cost of errors, and feedback loop before optimizing an algorithm. Distribution shift can make yesterday's validation result irrelevant, so production systems monitor inputs, predictions, outcomes, latency, and failures.

scikit-learn's estimator interface enables custom components. Estimators expose `fit`; predictors expose `predict`; transformers expose `transform`. Following this interface makes custom logic compatible with pipelines and cross-validation.

Human-in-the-loop systems route uncertain or high-impact cases to review. Automation should include an abstention policy rather than pretending every prediction is equally reliable.

**Book coverage:** approaching ML problems, humans in the loop, prototype-to-production concerns, testing, custom estimators, and next steps (Chapter 8, printed pp. 357-366).

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report

data = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, stratify=data.target, random_state=42)
production_candidate = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000))
production_candidate.fit(X_train, y_train)
print(classification_report(y_test, production_candidate.predict(X_test), target_names=data.target_names))

              precision    recall  f1-score   support

   malignant       0.98      0.98      0.98        53
      benign       0.99      0.99      0.99        90

    accuracy                           0.99       143
   macro avg       0.99      0.99      0.99       143
weighted avg       0.99      0.99      0.99       143



In [2]:
from sklearn.base import BaseEstimator, TransformerMixin

class PercentileClipper(TransformerMixin, BaseEstimator):
    # Clip every feature to bounds learned from the training set.
    def __init__(self, lower=1.0, upper=99.0):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        X = np.asarray(X)
        self.lower_bounds_ = np.percentile(X, self.lower, axis=0)
        self.upper_bounds_ = np.percentile(X, self.upper, axis=0)
        return self

    def transform(self, X):
        return np.clip(np.asarray(X), self.lower_bounds_, self.upper_bounds_)

custom_pipeline = make_pipeline(PercentileClipper(), StandardScaler(), LogisticRegression(max_iter=3000))
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(custom_pipeline, data.data, data.target, cv=cv)
print("Custom pipeline CV scores:", scores.round(3))
print(f"Mean +/- SD: {scores.mean():.3f} +/- {scores.std():.3f}")

Custom pipeline CV scores: [0.974 0.939 0.974 0.991 0.991]
Mean +/- SD: 0.974 +/- 0.019


In [3]:
probability = production_candidate.predict_proba(X_test)[:, 1]
confidence = np.maximum(probability, 1 - probability)
review_threshold = 0.80
automatic = confidence >= review_threshold

review_table = pd.DataFrame({
    "probability_benign": probability,
    "confidence": confidence,
    "decision": np.where(automatic, np.where(probability >= 0.5, "auto-benign", "auto-malignant"), "human-review"),
    "actual": data.target_names[y_test],
})
display(review_table.head(10).round(3))
print(f"Automatically handled: {automatic.mean():.1%}")
print(f"Sent to human review: {(~automatic).mean():.1%}")

,probability_benign,confidence,decision,actual
0,0.969,0.969,auto-benign,benign
1,0.000,1.000,auto-malignant,malignant
2,0.559,0.559,human-review,benign
3,0.939,0.939,auto-benign,benign
4,0.175,0.825,auto-malignant,malignant
5,0.000,1.000,auto-malignant,malignant
6,0.003,0.997,auto-malignant,malignant
7,0.001,0.999,auto-malignant,malignant
8,0.001,0.999,auto-malignant,malignant
9,1.000,1.000,auto-benign,benign


Automatically handled: 93.7%
Sent to human review: 6.3%


In [4]:
reference_mean = X_train.mean(axis=0)
incoming_mean = X_test.mean(axis=0)
reference_std = X_train.std(axis=0)
standardized_shift = np.abs(incoming_mean - reference_mean) / np.maximum(reference_std, 1e-12)
largest = np.argsort(standardized_shift)[-5:][::-1]
shift_report = pd.DataFrame({
    "feature": data.feature_names[largest],
    "standardized_mean_shift": standardized_shift[largest],
})
display(shift_report.round(3))
print("Illustrative alert rule: investigate features with standardized mean shift > 0.5")

,feature,standardized_mean_shift
0,mean symmetry,0.138
1,area error,0.112
2,symmetry error,0.111
3,mean smoothness,0.102
4,worst concavity,0.100


Illustrative alert rule: investigate features with standardized mean shift > 0.5


## Key Takeaways
- Define the decision, success metric, and error costs before modeling.
- Package every learned preprocessing step with the model.
- Test data contracts and custom components, not only model accuracy.
- Monitor drift and real outcomes after deployment.
- Route uncertain or consequential predictions to people.
- Reproducibility, documentation, privacy, and maintenance are part of model quality.